<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [106]:
%pip install langchain langchain_community pypdf
%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv duckduckgo-search

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\34673\AppData\Roaming\Python\Python314\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=13>
  res = process_handler(cmd, _system_body)
C:\Users\34673\AppData\Roaming\Python\Python314\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=11>
  res = process_handler(cmd, _system_body)
C:\Users\34673\AppData\Roaming\Python\Python314\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=12>
  res = process_handler(cmd, _system_body)


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\34673\AppData\Roaming\Python\Python314\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=13>
  res = process_handler(cmd, _system_body)
C:\Users\34673\AppData\Roaming\Python\Python314\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=11>
  res = process_handler(cmd, _system_body)
C:\Users\34673\AppData\Roaming\Python\Python314\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=12>
  res = process_handler(cmd, _system_body)


In [132]:
import langchain
print(langchain.__version__)

1.3.0


In [133]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from duckduckgo_search import DDGS
import warnings
warnings.filterwarnings("ignore")

<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [134]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [135]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [136]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [137]:
import os
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [138]:
load_dotenv()

True

In [139]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [140]:
from langchain_chroma import Chroma

In [141]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


### Web Search Fallback (DuckDuckGo)

In [143]:
# _get_document_prompt defined here so get_context can use it immediately
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

def web_search_fallback(query, max_results=3):
    """Fetch short snippets from DuckDuckGo - only called when the book has no relevant match."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=max_results))
    return results

def get_context(user_question, db, score_threshold=0.2):
    """
    Try the book vector store first.
    Falls back to DuckDuckGo only when NO chunk scores at or above the threshold.
    Prints scores so you can tune the threshold if needed.
    Returns: (retrieved_docs, formatted_context, source)
    """
    results = db.similarity_search_with_relevance_scores(user_question, k=10)

    # Show scores so you can tune the threshold
    print("Top relevance scores:")
    for doc, score in results[:5]:
        print(f"  {score:.4f} | {doc.page_content[:60].strip()}...")

    good_docs = [doc for doc, score in results if score >= score_threshold]

    if good_docs:
        print(f"\nSource: BOOK ({len(good_docs)} chunks above threshold {score_threshold})")
        context = _get_document_prompt(good_docs)
        return good_docs, context, "book"

    # Fallback: DuckDuckGo snippets (short = fewer tokens consumed)
    print(f"\nNo book chunks above {score_threshold} - falling back to DuckDuckGo.")
    web_results = web_search_fallback(user_question)
    context = "\n\n".join([
        f"Title: {r['title']}\nSnippet: {r['body'][:300]}\nURL: {r['href']}"
        for r in web_results
    ])
    return web_results, context, "web"

print("get_context and helpers ready.")

get_context and helpers ready.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercise1: Write a user question that someone might ask about your book’s topic or content.

In [144]:
user_question = "What are the key steps a company should take to build an AI strategy?"

# Uses book first; falls back to DuckDuckGo only if no relevant chunk is found
retrieved_docs, formatted_context, source = get_context(user_question, db)
print(f"Source used: {source.upper()} | Chunks retrieved: {len(retrieved_docs)}")

Top relevance scores:
  0.2837 | of AI development to their corporate benefit and the return...
  0.2837 | of AI development to their corporate benefit and the return...
  0.2837 | of AI development to their corporate benefit and the return...
  0.2837 | of AI development to their corporate benefit and the return...
  0.2837 | of AI development to their corporate benefit and the return...

Source: BOOK (10 chunks above threshold 0.2)
Source used: BOOK | Chunks retrieved: 10


In [145]:
# Display top results
for i, item in enumerate(retrieved_docs[:3]):
    if source == "book":
        print(f"Document {i+1}:\n{item.page_content[:400]}")
    else:
        print(f"Web Result {i+1}:\nTitle : {item['title']}\n"
              f"Snippet: {item['body'][:300]}\nURL    : {item['href']}")
    print('-'*80)

Document 1:
of AI development to their corporate benefit and the return on investment for 
their investors and shareholders?
The AI4People initiative, set up by the non-profit organisation Atomium-
EISMD (European Institute for Science, Media and Democracy), is the Euro -
pean response to the Asilomar AI initiative. AI4People also brings together 
academics, business partners (e.g. Facebook, Intel and Microso
--------------------------------------------------------------------------------
Document 2:
of AI development to their corporate benefit and the return on investment for 
their investors and shareholders?
The AI4People initiative, set up by the non-profit organisation Atomium-
EISMD (European Institute for Science, Media and Democracy), is the Euro -
pean response to the Asilomar AI initiative. AI4People also brings together 
academics, business partners (e.g. Facebook, Intel and Microso
--------------------------------------------------------------------------------
Document 3:


<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [146]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [147]:
# formatted_context is already built inside get_context() above
print(f"Context ready ({source}). Total characters: {len(formatted_context)}")

Context ready (book). Total characters: 9436


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [148]:
user_question

'What are the key steps a company should take to build an AI strategy?'

In [149]:
prompt = f"""
## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with questions about **Artificial Intelligence strategy and adoption**, specifically focusing on how **companies and individuals can leverage AI effectively**.
Your primary source is the book **"AI for Everyone" by Andrew Ng**.
If the answer is not found in the provided context, you may supplement your response with information from the web — but you must clearly distinguish between book content and web content.

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the relevant content from the book:
\'\'\'
{formatted_context}
\'\'\'

## GUIDELINES
1. **Accuracy**:
   - Prioritise the content in the `CONTEXT` section above all other sources.
   - If the answer is fully found in the context: answer using only the book.
   - If the answer is partially or not found in the context: supplement with web knowledge and clearly label it as such.
   - Structure your response as:
     1. Direct answer to the question.
     2. Key concepts broken into bullet points (e.g., strategic steps, tools, roles, timelines).
     3. Practical recommendations or examples.
     4. Any caveats or limitations the author or source mentions.

2. **Transparency**:
   - Always state whether information comes from the book or the web.
   - When citing the book: include the book name and page numbers.
   - When citing the web: include the full URL as a clickable link.
   - Do not speculate or present opinions as facts.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.
   - Target a non-technical business audience (executives, managers, decision-makers).

## TASK
1. Answer the user\'s question **directly** if possible.
2. Point the user to the relevant parts of the book and/or web sources.
3. Provide the response in the following format:

## RESPONSE FORMAT
\'\'\'
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Key Points**:
• [Point 1]
• [Point 2]
• [Point 3]

**Source**:
• AI for Everyone, Andrew Ng — Page(s): [...]       ← if from the book
• [Web Source Title] — [URL]                         ← if from the web
\'\'\'
"""

In [ ]:
"""
prompt = f"""
## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with questions about **Artificial Intelligence strategy and adoption**, specifically focusing on how **companies and individuals can leverage AI effectively**.
Your answers must be based exclusively on the provided content from the book **"AI for Everyone" by Andrew Ng**.

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the relevant content from the book:
\'\'\'
{formatted_context}
\'\'\'

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "The provided context does not contain this information."
   - Start by giving a direct answer to the question.
   - Then break down the key concepts in bullet points (e.g., strategic steps, tools, roles, timelines).
   - Follow with practical recommendations or examples from the book.
   - Lastly, note any caveats or limitations the author mentions.

2. **Transparency**:
   - Reference the book name and page numbers when providing information.
   - Do not speculate or provide opinions beyond what the book states.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.
   - Target a non-technical business audience (executives, managers, decision-makers).

## TASK
1. Answer the user\'s question **directly** if possible.
2. Point the user to the relevant parts of the book.
3. Provide the response in the following format:

## RESPONSE FORMAT
\'\'\'
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Key Points**:
• [Point 1]
• [Point 2]
• [Point 3]

**Source**:
• AI for Everyone, Andrew Ng — Page(s): [...]
\'\'\'
"""
"""

In [150]:
print(prompt)


## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with questions about **Artificial Intelligence strategy and adoption**, specifically focusing on how **companies and individuals can leverage AI effectively**.
Your primary source is the book **"AI for Everyone" by Andrew Ng**.
If the answer is not found in the provided context, you may supplement your response with information from the web — but you must clearly distinguish between book content and web content.

## USER QUESTION
The user has asked:
"What are the key steps a company should take to build an AI strategy?"

## CONTEXT
Here is the relevant content from the book:
'''


Content:
of AI development to their corporate benefit and the return on investment for 
their investors and shareholders?
The AI4People initiative, set up by the non-profit organisation Atomium-
EISMD (European Institute for Science, Media and Democracy), is the Euro -
pean response to the Asilomar AI initiative. AI4People also bri

In [151]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [152]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.7,   # Moderate creativity — factual but readable
    'max_tokens': 1000,    # Enough for a thorough answer
    'top_p': 0.9,         # Nucleus sampling for natural variation
    'frequency_penalty': 0.3,  # Reduce repetition of phrases
    'presence_penalty': 0.2    # Encourage covering new points
}

<h1 style="color: #FF6347;">Response</h1>


In [153]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [154]:
answer = completion.choices[0].message.content
print(answer)

# Key Steps to Build an AI Strategy

To build an effective AI strategy, companies should focus on aligning AI initiatives with their business objectives and ensuring ethical considerations are integrated into the process.

**Key Points**:
• **Define Business Goals**: Identify specific business problems that AI can solve, aligning AI projects with overall corporate strategy.
• **Develop Ethical Guidelines**: Establish ethical principles for AI use, such as those proposed by the AI4People initiative, which include beneficence and fairness.
• **Assess and Build Capabilities**: Evaluate current capabilities in data and technology infrastructure and invest in necessary tools and training for staff.

**Source**:
• AI for Everyone, Andrew Ng — Page(s): [Not specified in provided context]


<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [155]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [156]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

In [157]:
query_keywords = ["AI", "strategy", "company", "data", "machine learning",
                  "automation", "workflow", "CEO"]

if source == "book":
    for i, doc in enumerate(retrieved_docs[:1]):
        snippet = doc.page_content[:200]
        highlighted = highlight_keywords(snippet, query_keywords)
        print(f'Snippet {i+1}:\n{highlighted}\n' + '-'*80)
else:
    print("Web source used — highlighting skipped to preserve tokens.\n")
    for r in retrieved_docs[:1]:
        print(f"Title  : {r['title']}\nURL    : {r['href']}\nSnippet: {r['body'][:200]}")

Snippet 1:
of AI development to their corporate benefit and the return on investment for 
their investors and shareholders?
The AI4People initiative, set up by the non-profit organisation Atomium-
EISMD (Europea
--------------------------------------------------------------------------------


1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


### Step 1 — Load the new PDF

In [158]:
# Path to the bonus PDF (relative to this notebook)
bonus_file_path = "../How-to-win-friends-and-influence-people-Dale-Carnegie.pdf"

bonus_loader = PyPDFLoader(bonus_file_path)
bonus_pages = bonus_loader.load_and_split()
print(f"Total pages loaded: {len(bonus_pages)}")

Total pages loaded: 190


### Step 2 — Split pages into chunks

In [159]:
bonus_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
bonus_chunks = bonus_text_splitter.split_documents(bonus_pages)
print(f"Total chunks created: {len(bonus_chunks)}")

Total chunks created: 616


### Step 3 — Create embeddings and store in ChromaDB

In [160]:
# Stored in a separate collection so it does not mix with the AI for Everyone data
bonus_db = Chroma.from_documents(
    bonus_chunks,
    embeddings,
    persist_directory="./chroma_db_BONUS"
)
print("ChromaDB bonus collection created.")

ChromaDB bonus collection created.


### Step 4 — Retrieve relevant chunks

In [161]:
bonus_question = "What are the key principles for winning people over and building strong relationships?"

bonus_retrieved_docs = bonus_db.similarity_search(bonus_question, k=10)

# Preview top 3 retrieved chunks
for i, doc in enumerate(bonus_retrieved_docs[:3]):
    print(f'Document {i+1}:\n{doc.page_content[:300]}\n' + '-'*80)

Document 1:
Begin in a friendly way. 
 
PRINCIPLE 5 
Get the other person saying “yes, yes” immediately. 
 
PRINCIPLE 6 
Let the other person do a great deal of the talking. 
 
PRINCIPLE 7 
Let the other person feel that the idea is his or hers. 
 
PRINCIPLE 8 
Try honestly to see things from the other person's
--------------------------------------------------------------------------------
Document 2:
Begin in a friendly way. 
 
PRINCIPLE 5 
Get the other person saying “yes, yes” immediately. 
 
PRINCIPLE 6 
Let the other person do a great deal of the talking. 
 
PRINCIPLE 7 
Let the other person feel that the idea is his or hers. 
 
PRINCIPLE 8 
Try honestly to see things from the other person's
--------------------------------------------------------------------------------
Document 3:
In a Nutshell 
SIX WAYS TO MAKE PEOPLE LIKE YOU 
 
 
PRINCIPLE 1 
Become genuinely interested in other people. 
 
PRINCIPLE 2 
Smile. 
 
PRINCIPLE 3 
Remember that a person’s name is to that person th

### Step 5 — Format context for the model

In [162]:
bonus_formatted_context = _get_document_prompt(bonus_retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


### Step 6 — Build the prompt

In [163]:
bonus_question

'What are the key principles for winning people over and building strong relationships?'

In [164]:
bonus_prompt = f"""
## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with questions about **interpersonal skills, communication, and influence**, specifically based on the book **'How to Win Friends and Influence People' by Dale Carnegie**.
Your answers must be based exclusively on the provided content from the book.

## USER QUESTION
The user has asked:
"{bonus_question}"

## CONTEXT
Here is the relevant content from the book:
\'\'\'\n
{bonus_formatted_context}
\'\'\'\n

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: \"The provided context does not contain this information.\"
   - Start with a direct answer to the question.
   - Break down Carnegie's key principles in bullet points.
   - Follow with practical application tips from the book.
   - End with any caveats or context the author provides.

2. **Transparency**:
   - Reference the book name and page numbers when providing information.
   - Do not speculate or provide opinions beyond what the book states.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.
   - Target a general audience interested in self-improvement and leadership.

## TASK
1. Answer the user's question **directly** if possible.
2. Point the user to relevant parts of the book.
3. Provide the response in the following format:

## RESPONSE FORMAT
\'\'\'\n
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Key Principles**:
• [Principle 1]
• [Principle 2]
• [Principle 3]

**Source**:
• How to Win Friends and Influence People, Dale Carnegie — Page(s): [...]
\'\'\'\n
"""

In [165]:
print(bonus_prompt)


## SYSTEM ROLE
You are a knowledgeable and factual chatbot designed to assist with questions about **interpersonal skills, communication, and influence**, specifically based on the book **'How to Win Friends and Influence People' by Dale Carnegie**.
Your answers must be based exclusively on the provided content from the book.

## USER QUESTION
The user has asked:
"What are the key principles for winning people over and building strong relationships?"

## CONTEXT
Here is the relevant content from the book:
'''



Content:
Begin in a friendly way. 
 
PRINCIPLE 5 
Get the other person saying “yes, yes” immediately. 
 
PRINCIPLE 6 
Let the other person do a great deal of the talking. 
 
PRINCIPLE 7 
Let the other person feel that the idea is his or hers. 
 
PRINCIPLE 8 
Try honestly to see things from the other person's point of view. 
 
PRINCIPLE 9 
Be sympathetic with the other person's ideas and desires. 
 
PRINCIPLE 10 
Appeal to the nobler motives. 
 
PRINCIPLE 11 
Dramatize your ide

### Step 7 — Generate the response

In [166]:
bonus_model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.7,
    'max_tokens': 512,
    'top_p': 0.9,
    'frequency_penalty': 0.3,
    'presence_penalty': 0.2
}

bonus_messages = [{'role': 'user', 'content': bonus_prompt}]
bonus_completion = client.chat.completions.create(
    messages=bonus_messages,
    **bonus_model_params,
    timeout=120
)

bonus_answer = bonus_completion.choices[0].message.content
print(bonus_answer)

# Key Principles for Winning People Over

To win people over and build strong relationships, Dale Carnegie outlines several key principles that focus on understanding and valuing others.

**Key Principles**:
• Begin in a friendly way.  
• Become genuinely interested in other people.  
• Smile.  
• Remember that a person’s name is to that person the sweetest sound in any language.  
• Be a good listener; encourage others to talk about themselves.  
• Talk in terms of the other person’s interests.  
• Make the other person feel important—and do it sincerely.  
• Get the other person saying “yes, yes” immediately.  
• Let the other person do a great deal of the talking.  
• Let the other person feel that the idea is his or hers.  
• Try honestly to see things from the other person's point of view.  
• Be sympathetic with the other person's ideas and desires.  
• Appeal to the nobler motives.  
• Dramatize your ideas.  
• Throw down a challenge.

**Practical Application Tips**:
1. **Show G

### Step 8 — Keyword highlighting

In [167]:
from termcolor import colored

In [168]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

In [169]:
bonus_keywords = ["friends", "friendly", "people", "influence", "principles", "smile", "name", "listen", "appreciation", "criticism", "leadership"]

for i, doc in enumerate(bonus_retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, bonus_keywords)
    print(f'Snippet {i+1}:\n{highlighted}\n' + '-'*80)

Snippet 1:
Begin in a friendly way. 
 
PRINCIPLE 5 
Get the other person saying “yes, yes” immediately. 
 
PRINCIPLE 6 
Let the other person do a great deal of the talking. 
 
PRINCIPLE 7 
Let the other person f
--------------------------------------------------------------------------------
